# Data Cleaning -- Training Dataset
**This notebook prepares Flatiron Health data for patients with metastatic colorectal cancer in anticipation of training a gradient-boosted survival model to predict mortality from the start of first-line treatment. Specifically, it processes and cleans the training cohort. Prior to data cleaning, the dataset is randomly split into training (80%) and testing (20%) subsets.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from flatiron_cleaner import DataProcessorColorectal, merge_dataframes

In [2]:
# Function that returns number of rows and count of unique PatientIDs for a dataframe. 
def row_ID(dataframe):
    row = dataframe.shape[0]
    ID = dataframe['PatientID'].nunique()
    return row, ID

## Split into training sets

In [3]:
df = pd.read_csv('../data/LineOfTherapy.csv')

In [4]:
df = (
    df
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    [['PatientID', 'StartDate']]
)

In [5]:
row_ID(df)

(38264, 37425)

In [6]:
df['StartDate'] = pd.to_datetime(df['StartDate'])

In [7]:
# Remove duplicate PatientIDs and select first for each patient based on StartDate
df = (
    df
    .sort_values(['PatientID', 'StartDate'], ascending = True)
    .drop_duplicates(subset='PatientID', keep='first')
)

In [8]:
df.shape

(37425, 2)

In [9]:
processor = DataProcessorColorectal()

In [10]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_MetCRCBiomarkers.csv',
                                           her2_path = '../data/Enhanced_CRC_HER2.csv',
                                           oral_path = '../data/Enhanced_MetCRC_Orals.csv', 
                                           progression_path = '../data/Enhanced_MetCRC_Progression.csv',
                                           drop_dates = False)

2026-03-21 20:02:25,955 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (31065, 2) and unique PatientIDs: 31065
2026-03-21 20:02:25,982 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (37425, 3) and unique PatientIDs: 37425
2026-03-21 20:02:28,533 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_her2_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2026-03-21 20:02:28,552 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (37425, 6) and unique PatientIDs: 37425. There are 0 out of 37425 patients with missing duration values


In [11]:
df = pd.merge(df, mortality_df[['PatientID', 'event']], on = 'PatientID', how = 'left')

In [12]:
df.shape

(37425, 3)

In [13]:
# Stratify split on event 
train, test = train_test_split(
    df,
    test_size = 0.2,
    stratify = df['event'],  
    random_state = 42
)

In [14]:
row_ID(train)

(29940, 29940)

In [15]:
row_ID(test)

(7485, 7485)

In [16]:
train[['PatientID']].to_csv('../outputs/train_patient_ids.csv', index = False)
test[['PatientID']].to_csv('../outputs/test_patient_ids.csv', index = False)

## Clean CSV files 

In [17]:
train_ids = pd.read_csv('../outputs/train_patient_ids.csv')

In [18]:
train_ids = train_ids.PatientID.to_list()

In [19]:
index_date_df = df[df.PatientID.isin(train_ids)]

In [20]:
index_date_df.shape

(29940, 3)

In [21]:
index_date_df = index_date_df[['PatientID', 'StartDate']]

### Process Enhanced_MetastaticCRC.csv

In [22]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_MetastaticCRC.csv',
                                         patient_ids = train_ids,
                                         drop_stage = True,
                                         drop_dates = False)

2026-03-21 20:02:28,713 - INFO - Successfully read Enhanced_MetastaticCRC.csv file with shape: (46877, 5) and unique PatientIDs: 46877
2026-03-21 20:02:28,714 - INFO - Filtering for 29940 specific PatientIDs
2026-03-21 20:02:28,722 - INFO - Successfully filtered Enhanced_MetastaticCRC.csv file with shape: (29940, 5) and unique PatientIDs: 29940
2026-03-21 20:02:28,741 - INFO - Successfully processed Enhanced_MetastaticCRC.csv file with final shape: (29940, 7) and unique PatientIDs: 29940


In [23]:
enhanced_df.shape

(29940, 7)

In [24]:
enhanced_df = pd.merge(enhanced_df, index_date_df[['PatientID', 'StartDate']], on = 'PatientID', how = 'left')

In [25]:
enhanced_df.shape

(29940, 8)

In [26]:
enhanced_df['days_met_to_treatment'] = (enhanced_df['StartDate'] - enhanced_df['MetDiagnosisDate']).dt.days
enhanced_df['days_met_to_treatment'] = np.where(enhanced_df['days_met_to_treatment'] < 0 , 0, enhanced_df['days_met_to_treatment'])

In [27]:
enhanced_df['days_diagnosis_to_met'] = np.where(enhanced_df['days_diagnosis_to_met'] < 0 , 0, enhanced_df['days_diagnosis_to_met'])
enhanced_df['days_diagnosis_to_met'] = enhanced_df['days_diagnosis_to_met'].fillna(0)

In [28]:
enhanced_df.GroupStage_mod.value_counts(dropna = False)

GroupStage_mod
IV         18309
III         6858
II          2991
unknown      999
I            778
0              5
Name: count, dtype: int64

In [29]:
enhanced_df['GroupStage_mod'] = enhanced_df["GroupStage_mod"].map({
    '0': 0, 
    'I': 1,
    'II': 2,
    'III': 3,
    'IV': 4,
    'unknown': np.nan
})

enhanced_df['GroupStage_mod_na'] = np.where(enhanced_df['GroupStage_mod'].isna(), 1, 0)

# impute 4 since stage IV is most common
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].fillna(4)

In [30]:
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate', 
                                          'MetDiagnosisDate', 
                                          'StartDate'])

### Process Demographics.csv 

In [31]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = index_date_df,
                                                 index_date_column = 'StartDate')

2026-03-21 20:02:28,802 - INFO - Successfully read Demographics.csv file with shape: (46877, 6) and unique PatientIDs: 46877
2026-03-21 20:02:28,822 - WARNING - Found 1 ages outside valid range (18-120)
2026-03-21 20:02:28,840 - INFO - Successfully processed Demographics.csv file with final shape: (29940, 6) and unique PatientIDs: 29940


In [32]:
demographics_df = demographics_df.copy()

In [33]:
demographics_df = demographics_df.query('age >= 18 and age <= 120', engine = 'python')

In [34]:
demographics_df.Gender.value_counts(dropna = False)

Gender
M      16655
F      13278
NaN        6
Name: count, dtype: int64

In [35]:
# Impute missing with most common sex (male)
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [36]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_MetCRCBiomarkers.csv

In [37]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_MetCRCBiomarkers.csv',
                                             index_date_df = index_date_df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2026-03-21 20:02:29,089 - INFO - Successfully read Enhanced_MetCRCBiomarkers.csv file with shape: (207027, 15) and unique PatientIDs: 42112
2026-03-21 20:02:29,187 - INFO - Successfully merged Enhanced_MetCRCBiomarkers.csv df with index_date_df resulting in shape: (144888, 16) and unique PatientIDs: 27969
2026-03-21 20:02:29,667 - INFO - Successfully processed Enhanced_MetCRCBiomarkers.csv file with final shape: (29940, 5) and unique PatientIDs: 29940


In [38]:
biomarkers_df['BRAF_status'] = biomarkers_df['BRAF_status'].fillna('unknown')
biomarkers_df['KRAS_status'] = biomarkers_df['KRAS_status'].fillna('unknown')
biomarkers_df['NRAS_status'] = biomarkers_df['NRAS_status'].fillna('unknown')
biomarkers_df['MMR/MSI_status'] = biomarkers_df['MMR/MSI_status'].fillna('unknown')

In [39]:
biomarkers_df['MMR/MSI_status'].value_counts(dropna = False)

MMR/MSI_status
negative    18161
unknown     10403
positive     1376
Name: count, dtype: int64

In [40]:
biomarkers_df['BRAF_status'].value_counts(dropna = False)

BRAF_status
unknown     14695
negative    13579
positive     1666
Name: count, dtype: int64

### Process Enhanced_CRC_HER2.csv

In [41]:
her2_df = processor.process_her2(file_path = '../data/Enhanced_CRC_HER2.csv',
                                 index_date_df = index_date_df, 
                                 index_date_column = 'StartDate',
                                 days_before = None, 
                                 days_after = 30)

2026-03-21 20:02:29,715 - INFO - Successfully read Enhanced_CRC_HER2.csv file with shape: (31958, 8) and unique PatientIDs: 18810
2026-03-21 20:02:29,735 - INFO - Successfully merged Enhanced_CRC_HER2.csv df with index_date_df resulting in shape: (23007, 9) and unique PatientIDs: 13375
2026-03-21 20:02:29,857 - INFO - Successfully processed Enhanced_CRC_HER2.csv file with final shape: (29940, 3) and unique PatientIDs: 29940


In [42]:
her2_df['HER2_status'] = her2_df['HER2_status'].fillna('unknown')

In [43]:
her2_df['HER2_percent_staining'] = her2_df['HER2_percent_staining'].cat.add_categories(['unknown'])
her2_df['HER2_percent_staining'] = her2_df['HER2_percent_staining'].fillna('unknown')

In [44]:
her2_df.HER2_status.value_counts(dropna = False)

HER2_status
unknown     21074
negative     8273
positive      593
Name: count, dtype: int64

In [45]:
her2_df.HER2_percent_staining.value_counts(dropna = False)

HER2_percent_staining
unknown      29906
90% - 99%        8
100%             7
10% - 19%        6
40% - 49%        4
50% - 59%        3
70% - 79%        3
0%               1
60% - 69%        1
80% - 89%        1
< 1%             0
1%               0
2% - 4%          0
5% - 9%          0
20% - 29%        0
30% - 39%        0
Name: count, dtype: int64

### Process ECOG.csv

In [46]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-03-21 20:02:30,254 - INFO - Successfully read ECOG.csv file with shape: (1017535, 4) and unique PatientIDs: 36246
2026-03-21 20:02:30,585 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (777265, 5) and unique PatientIDs: 24597
2026-03-21 20:02:30,898 - INFO - Successfully processed ECOG.csv file with final shape: (29940, 3) and unique PatientIDs: 29940


In [47]:
ecog_df.ecog_index.value_counts(dropna = False)

ecog_index
NaN    10550
0       8853
1       7904
2       2122
3        475
4         36
Name: count, dtype: int64

In [48]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 0 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(0)

In [49]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [50]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = index_date_df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2026-03-21 20:03:16,784 - INFO - Successfully read Vitals.csv file with shape: (17264775, 16) and unique PatientIDs: 46724
2026-03-21 20:03:27,635 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (12857431, 17) and unique PatientIDs: 29915
2026-03-21 20:03:32,358 - INFO - Successfully processed Vitals.csv file with final shape: (29940, 8) and unique PatientIDs: 29940


### Process Lab.csv

In [51]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 additional_loinc_mappings = {'cea': ['2039-6'], 'ldh': ['2532-0', '14804-9']},
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-03-21 20:04:49,738 - INFO - Successfully read Lab.csv file with shape: (47135286, 17) and unique PatientIDs: 44960
2026-03-21 20:05:27,390 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (35399687, 18) and unique PatientIDs: 29343
2026-03-21 20:06:39,075 - INFO - Successfully processed Lab.csv file with final shape: (29940, 86) and unique PatientIDs: 29940


### Process MedicationAdministration.csv

In [52]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = index_date_df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2026-03-21 20:06:48,619 - INFO - Successfully read MedicationAdministration.csv file with shape: (6618222, 11) and unique PatientIDs: 38657
2026-03-21 20:06:51,755 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (5004812, 12) and unique PatientIDs: 27868
2026-03-21 20:06:52,023 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (29940, 9) and unique PatientIDs: 29940


### Process Diagnosis.csv

In [53]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = index_date_df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2026-03-21 20:06:54,507 - INFO - Successfully read Diagnosis.csv file with shape: (3378971, 6) and unique PatientIDs: 46877
2026-03-21 20:06:55,485 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (2388642, 7) and unique PatientIDs: 29940
2026-03-21 20:06:59,977 - INFO - Successfully processed Diagnosis.csv file with final shape: (29940, 40) and unique PatientIDs: 29940


### Process Enhanced_Mortality_V2.csv

In [54]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = index_date_df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_MetCRCBiomarkers.csv',
                                           her2_path = '../data/Enhanced_CRC_HER2.csv',
                                           oral_path = '../data/Enhanced_MetCRC_Orals.csv', 
                                           progression_path = '../data/Enhanced_MetCRC_Progression.csv',
                                           drop_dates = False)

2026-03-21 20:07:00,041 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (31065, 2) and unique PatientIDs: 31065
2026-03-21 20:07:00,069 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (29940, 3) and unique PatientIDs: 29940
2026-03-21 20:07:02,570 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_her2_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2026-03-21 20:07:02,589 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (29940, 6) and unique PatientIDs: 29940. There are 0 out of 29940 patients with missing duration values


In [55]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [56]:
mortality_df = mortality_df.query('duration >= 0')

## Merge dataframes

In [57]:
training_df = merge_dataframes(enhanced_df,
                               demographics_df,
                               biomarkers_df,
                               her2_df,
                               ecog_df,
                               vitals_df,
                               labs_df,
                               medications_df,
                               diagnosis_df,
                               mortality_df,
                               merge_type = 'inner')

2026-03-21 20:07:02,674 - INFO - Anticipated number of merges: 9
2026-03-21 20:07:02,675 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 162
2026-03-21 20:07:02,679 - INFO - Dataset 1 shape: (29940, 7), unique PatientIDs: 29940
2026-03-21 20:07:02,684 - INFO - Dataset 2 shape: (29939, 6), unique PatientIDs: 29939
2026-03-21 20:07:02,688 - INFO - Dataset 3 shape: (29940, 5), unique PatientIDs: 29940
2026-03-21 20:07:02,694 - INFO - Dataset 4 shape: (29940, 3), unique PatientIDs: 29940
2026-03-21 20:07:02,698 - INFO - Dataset 5 shape: (29940, 4), unique PatientIDs: 29940
2026-03-21 20:07:02,703 - INFO - Dataset 6 shape: (29940, 8), unique PatientIDs: 29940
2026-03-21 20:07:02,707 - INFO - Dataset 7 shape: (29940, 86), unique PatientIDs: 29940
2026-03-21 20:07:02,711 - INFO - Dataset 8 shape: (29940, 9), unique PatientIDs: 29940
2026-03-21 20:07:02,714 - INFO - Dataset 9 shape: (29940, 40), unique PatientIDs: 29940
2026-03-2

In [58]:
training_df.shape

(29907, 162)

## Export dataframe

In [59]:
training_df.to_csv('../outputs/1L_features_training.csv', index = False)

In [60]:
# Save dtypes
training_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/1L_features_training_dtypes.csv')